# **Titanic Data Cleaning and Analysis Project**

**Dataset:** Titanic Passenger Dataset

**Objective:**
The goal of this project is to clean and prepare the Titanic dataset using Python and Pandas. The cleaned dataset will later be used in Power BI to create visualizations and gain insights about passenger survival patterns.

# **1. Import Required Dataset and Libraries**

I imported the required dataset and the libraries needed for data manipulation and numerical operations.

In [204]:
from google.colab import files
uploaded = files.upload()

Saving titanic.csv to titanic (3).csv


In [205]:
# Import pandas for data analysis and manipulation
import pandas as pd

# Import numpy for numerical operations and handling missing values
import numpy as np

# **2. Load the Dataset**

The dataset is loaded into a pandas DataFrame so that i can explore and clean it.

In [206]:
# Load the Titanic dataset into a DataFrame
df = pd.read_csv("titanic.csv")

# Display the first 5 rows of the dataset
df.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22,1,0,A/5 21171,7.2500,\N,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26,0,0,STON/O2. 3101282,7.9250,\N,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35,0,0,373450,8.0500,\N,S


# **3. Explore the Dataset**

Before cleaning the data, it is important to understand its structure, data types, and potential issues.

In [207]:
# Display summary information about the dataset
# This includes column names, data types, and number of non-null values
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 12 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  891 non-null    int64  
 1   Survived     891 non-null    int64  
 2   Pclass       891 non-null    int64  
 3   Name         891 non-null    object 
 4   Sex          891 non-null    object 
 5   Age          891 non-null    object 
 6   SibSp        891 non-null    int64  
 7   Parch        891 non-null    int64  
 8   Ticket       891 non-null    object 
 9   Fare         891 non-null    float64
 10  Cabin        891 non-null    object 
 11  Embarked     891 non-null    object 
dtypes: float64(1), int64(5), object(6)
memory usage: 83.7+ KB


In [208]:
# To check for missing values in each column
df.isnull().sum()

,0
PassengerId,0
Survived,0
Pclass,0
Name,0
Sex,0
Age,0
SibSp,0
Parch,0
Ticket,0
Fare,0


In [209]:
# To check if there are duplicate rows in the dataset
df.duplicated().sum()

np.int64(0)

# **4. Column Standardization**

I standardized the column names by converting them to lowercase and removing extra spaces. I did this to avoid errors when calling the columns in my analysis since the original dataset had mixed capitalization. This helps me maintain consistency while working with the dataset.

In [210]:
# Convert all column names to lowercase and remove extra spaces
df.columns = df.columns.str.lower().str.strip()

In [211]:
# This shows if all columns are converted to lowercase and extra spaces removed.
df.columns

Index(['passengerid', 'survived', 'pclass', 'name', 'sex', 'age', 'sibsp',
       'parch', 'ticket', 'fare', 'cabin', 'embarked'],
      dtype='object')

# **5. Replace Non-Standard Missing Values**

In this dataset, some missing values are represented as \N instead of NaN.
These must be replaced with proper NaN values so pandas can detect them.

In [212]:
df['age'].dtype

dtype('O')

In [213]:
df.columns

Index(['passengerid', 'survived', 'pclass', 'name', 'sex', 'age', 'sibsp',
       'parch', 'ticket', 'fare', 'cabin', 'embarked'],
      dtype='object')

In [214]:
import numpy as np
df['age'] = df['age'].replace('\\N', np.nan)

In [215]:
df.isnull().sum()

,0
passengerid,0
survived,0
pclass,0
name,0
sex,0
age,177
sibsp,0
parch,0
ticket,0
fare,0


# **6. Clean the Age Column**

The Age column contains missing values and must be converted to numeric format before filling missing values.

In [216]:
# Convert the Age column to numeric values
# Any invalid entries will be converted to NaN
df['age'] = pd.to_numeric(df['age'], errors='coerce')

In [217]:
# Fill missing Age values using the median age
# Median is preferred because it is less affected by outliers
df['age'] = df['age'].fillna(df['age'].median())

In [218]:
# To check if missing values are still present
df['age'].isnull().sum()

np.int64(0)

In [219]:
df.isnull().sum()

,0
passengerid,0
survived,0
pclass,0
name,0
sex,0
age,0
sibsp,0
parch,0
ticket,0
fare,0


# **7. Handle Cabin Column**

The Cabin column contains a large number of missing values.
Instead of filling them, we create a new column indicating whether a passenger had a cabin.

In [220]:
# Check if Cabin column exists before using it
if 'cabin' in df.columns:

    # Create a binary column: 1 if passenger had a cabin, 0 if not
    df['has_cabin'] = df['cabin'].notnull().astype(int)

    # Drop the original Cabin column because it has too many missing values
    df.drop(columns=['cabin'], inplace=True)

In [221]:
df.columns

Index(['passengerid', 'survived', 'pclass', 'name', 'sex', 'age', 'sibsp',
       'parch', 'ticket', 'fare', 'embarked', 'has_cabin'],
      dtype='object')

In [222]:
df.isnull().sum()

,0
passengerid,0
survived,0
pclass,0
name,0
sex,0
age,0
sibsp,0
parch,0
ticket,0
fare,0


# **8. Handle Embarked Column**

The Embarked column has a small number of missing values.
I replace them with the most frequent value (mode).

In [223]:
# Fill missing values in Embarked column using the most common value
df['embarked'] = df['embarked'].fillna(df['embarked'].mode()[0])

In [224]:
df.isnull().sum()

,0
passengerid,0
survived,0
pclass,0
name,0
sex,0
age,0
sibsp,0
parch,0
ticket,0
fare,0


# **9. Feature Engineering**

Feature engineering involves creating new useful variables from existing data.

**Create Family Size Feature**

This feature represents the total number of family members traveling together.

In [225]:
# Create a new column representing total family size
# It includes the passenger plus siblings/spouses and parents/children
df['family_size'] = df['sibsp'] + df['parch'] + 1

In [226]:
# To check if the new family_size column was calculated correctly.
df[['sibsp','parch','family_size']].head()

,sibsp,parch,family_size
0,1,0,2
1,1,0,2
2,0,0,1
3,1,0,2
4,0,0,1


**Create Age Group Feature**

Passengers are categorized into age groups for easier analysis.

In [227]:
# Define a function to categorize passengers into age groups
def age_group(age):
    if age < 13:
        return "Child"
    elif age < 20:
        return "Teen"
    elif age < 60:
        return "Adult"
    else:
        return "Senior"

# Apply the function to create a new age_group column
df['age_group'] = df['age'].apply(age_group)

In [228]:
# Display the first rows of age and age_group to confirm it worked
df[['age','age_group']].head()

,age,age_group
0,22.0,Adult
1,38.0,Adult
2,26.0,Adult
3,35.0,Adult
4,35.0,Adult


**Extract Passenger Title**

Passenger titles such as Mr, Mrs, and Miss can provide additional insights

In [229]:
df['title'] = df['name'].str.extract(r' ([A-Za-z]+)\.', expand=False)

In [230]:
# To check if the titles were extracted.
df[['name','title']].head()

,name,title
0,"Braund, Mr. Owen Harris",Mr
1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",Mrs
2,"Heikkinen, Miss. Laina",Miss
3,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",Mrs
4,"Allen, Mr. William Henry",Mr


# **10. Remove Duplicate Rows**

I removed the duplicates because it can affect analysis and should be removed.

In [231]:
df.drop_duplicates(inplace=True)

In [232]:
# To check if the duplicates were removed.
df.columns

Index(['passengerid', 'survived', 'pclass', 'name', 'sex', 'age', 'sibsp',
       'parch', 'ticket', 'fare', 'embarked', 'has_cabin', 'family_size',
       'age_group', 'title'],
      dtype='object')

# **11. Save Cleaned Dataset**

The cleaned dataset is exported as a CSV file to be used in Power BI for visualization.

In [233]:
# Save the cleaned dataset as a CSV file
df.to_csv("titanic_cleaned.csv", index=False, na_rep="")


# Download dataset
from google.colab import files
files.download("titanic_cleaned.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>